# Kansas City Education Labor Observatory: Focal District Baseline (Phase 1)

This notebook establishes the 10-year comparative baseline (2014–15 to 2023–24) across the **seven focal school districts** in the Greater Kansas City metropolitan area, comparing institutional, staffing, and governance dimensions across the Kansas–Missouri state line.

### The Focal 7 Panel Archetypes:
1. **USD 500 Kansas City, KS (KCKPS)** — *KS PNA / Strong Board Management Rights*
2. **USD 512 Shawnee Mission (SMSD)** — *KS PNA / Protected Planning Time & Compensation Benchmark*
3. **USD 233 Olathe** — *KS PNA / Collaborative Professional Council (6+6)*
4. **Kansas City Public Schools (KCPS / Missouri 33)** — *MO AFT Local 691 / Restricted Binding Arbitration*
5. **Independence School District (ISD 30)** — *MO NEA / Epicenter of 2007 Landmark Constitutional Precedent*
6. **North Kansas City Schools (NKC 74)** — *MO NEA / Collaborative Team for Teacher Negotiations & State Funding Triggers*
7. **Park Hill School District** — *MO NEA / Standing Policy Committees & Board Final Authority*

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path('../data/processed/focal_7_labor_panel.csv')
df = pd.read_csv(data_path)
print(f"Loaded {len(df)} records across {df['school_year'].nunique()} years for {df['focal_district_code'].nunique()} focal districts.")
df[['school_year', 'focal_district_code', 'state', 'enrollment_k12', 'teachers_k12_fte']].head(7)

## 1. Clean Pre-Break Benchmark: 2014–15 vs. 2023–24

We examine the 10-year changes in Enrollment, Classroom Teacher FTE, School Administrators, and Instructional Coordinators.

In [ ]:
years = ['2014-2015', '2023-2024']
df_comp = df[df['school_year'].isin(years)].copy()

piv = df_comp.pivot(index=['focal_district_code', 'state', 'focal_clean_name'], columns='school_year', 
                    values=['enrollment_k12', 'teachers_k12_fte', 'school_administrators_fte', 'instructional_coordinators_fte'])

# Calculate 10-year percentage changes
piv_summary = pd.DataFrame({
    'State': piv.index.get_level_values('state'),
    'District': piv.index.get_level_values('focal_clean_name'),
    'Enrollment 2015': piv[('enrollment_k12', '2014-2015')],
    'Enrollment 2024': piv[('enrollment_k12', '2023-2024')],
    'Enrollment % Chg': ((piv[('enrollment_k12', '2023-2024')] - piv[('enrollment_k12', '2014-2015')]) / piv[('enrollment_k12', '2014-2015')] * 100).round(1),
    'Teachers 2015': piv[('teachers_k12_fte', '2014-2015')],
    'Teachers 2024': piv[('teachers_k12_fte', '2023-2024')],
    'Teachers % Chg': ((piv[('teachers_k12_fte', '2023-2024')] - piv[('teachers_k12_fte', '2014-2015')]) / piv[('teachers_k12_fte', '2014-2015')] * 100).round(1),
    'Bldg Admin % Chg': ((piv[('school_administrators_fte', '2023-2024')] - piv[('school_administrators_fte', '2014-2015')]) / piv[('school_administrators_fte', '2014-2015')] * 100).round(1),
    'Coordinators % Chg': ((piv[('instructional_coordinators_fte', '2023-2024')] - piv[('instructional_coordinators_fte', '2014-2015')]) / piv[('instructional_coordinators_fte', '2014-2015')] * 100).round(1),
})
piv_summary.sort_values('Enrollment 2024', ascending=False)

## 2. Staffing Ratios & Supervisory Intensity (2023–24 Snapshot)

How many pupils per classroom teacher exist across these districts, and how many supervisors/coordinators exist per 100 teachers?

In [ ]:
latest = df[df['school_year'] == '2023-2024'].copy()
ratio_table = latest[[
    'focal_district_code', 'state', 'union_name', 'enrollment_k12', 
    'pupil_teacher_ratio', 'coordinators_per_100_teachers', 
    'core_admin_per_100_teachers', 'total_admin_coord_per_1000_students'
]].sort_values('pupil_teacher_ratio')

ratio_table.columns = [
    'Code', 'State', 'Bargaining Agent', 'K-12 Pupils', 
    'Pupil/Teacher Ratio', 'Coordinators / 100 Tchs', 
    'Admins / 100 Tchs', 'Supervisory / 1k Pupils'
]
ratio_table

## 3. Structural Takeaways for the Labor Observatory

1. **Pupil-Teacher Ratio Wedge:** KCPS exhibits the lowest pupil/teacher ratio (11.9:1), driven by urban core enrollment contraction (-10.4%) and classroom staffing baselines, while ISD 30 and KCKPS operate near 14.8:1.
2. **Supervisory & Coaching Differences:** Shawnee Mission and KCKPS maintain enormous coaching footprints (6.6 and 7.9 coordinators per 100 teachers), whereas Park Hill maintains only 0.5 per 100 teachers, reflecting divergent institutional theories of instructional oversight.
3. **Next Step:** Incorporate operative collective bargaining agreement clauses into `contract_clause_panel.parquet` to link these staffing structures directly to contract rules on planning time, management rights, and grievance enforcement.